# Instacart Lakehouse Analytics — Data Quality Checks

## Objective

This notebook validates the Silver and Gold layers using reusable data-quality checks.

The checks cover uniqueness, completeness, valid business ranges, referential integrity, and the validity of selected Gold-layer metrics.

## Inputs

- `silver_orders`
- `silver_product_dimension`
- `silver_order_items`
- `gold_customer_summary`
- `gold_product_performance`
- `gold_department_summary`
- `gold_order_behavior`

## Output

- `gold_data_quality_report`

## 1. Load Silver and Gold Tables

Load the curated Silver and Gold Delta tables from Unity Catalog for validation.

In [0]:
from pyspark.sql import functions as F

silver_orders = spark.table("workspace.default.silver_orders")
silver_products = spark.table("workspace.default.silver_product_dimension")
silver_items = spark.table("workspace.default.silver_order_items")

gold_customer = spark.table("workspace.default.gold_customer_summary")
gold_product = spark.table("workspace.default.gold_product_performance")
gold_department = spark.table("workspace.default.gold_department_summary")
gold_order_behavior = spark.table("workspace.default.gold_order_behavior")

## 2. Define a Reusable Quality-Check Framework

Create a helper function that records the check name, failed-row count, status, and severity.

A check passes when no failed rows are found.

In [0]:
# Store all data-quality results in a consistent structure
quality_results = []

def add_check(check_name, failed_rows, severity="ERROR"):
    status = "PASS" if failed_rows == 0 else "FAIL"

    quality_results.append(
        {
            "check_name": check_name,
            "failed_rows": int(failed_rows),
            "status": status,
            "severity": severity
        }
    )

## 3. Validate Key Uniqueness

Check that order and product identifiers remain unique in their respective Silver tables.

In [0]:
duplicate_orders = (
    silver_orders
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

add_check("Duplicate order IDs", duplicate_orders)

duplicate_products = (
    silver_products
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

add_check("Duplicate product IDs", duplicate_products)

## 4. Validate Required Fields

Check for missing values in essential identifiers and descriptive product attributes.

In [0]:
null_order_ids = silver_orders.filter(
    F.col("order_id").isNull()
).count()

add_check("Null order IDs", null_order_ids)

null_user_ids = silver_orders.filter(
    F.col("user_id").isNull()
).count()

add_check("Null user IDs", null_user_ids)

null_product_ids = silver_items.filter(
    F.col("product_id").isNull()
).count()

add_check("Null product IDs in order items", null_product_ids)

null_product_names = silver_items.filter(
    F.col("product_name").isNull()
).count()

add_check("Missing product names", null_product_names)

## 5. Validate Business Rules and Ranges

Confirm that key fields remain within expected business ranges, including:

- reordered values of 0 or 1
- weekday values from 0 through 6
- order-hour values from 0 through 23
- non-negative days between orders
- positive cart positions

In [0]:
invalid_reordered = silver_items.filter(
    ~F.col("reordered").isin(0, 1)
).count()

add_check("Invalid reordered values", invalid_reordered)

invalid_day_of_week = silver_orders.filter(
    (F.col("order_dow") < 0) |
    (F.col("order_dow") > 6)
).count()

add_check("Invalid day-of-week values", invalid_day_of_week)

invalid_hour = silver_orders.filter(
    (F.col("order_hour_of_day") < 0) |
    (F.col("order_hour_of_day") > 23)
).count()

add_check("Invalid order-hour values", invalid_hour)

negative_days_between_orders = silver_orders.filter(
    F.col("days_since_prior_order") < 0
).count()

add_check(
    "Negative days since prior order",
    negative_days_between_orders
)

invalid_cart_position = silver_items.filter(
    F.col("add_to_cart_order") <= 0
).count()

add_check("Invalid cart positions", invalid_cart_position)

## 6. Validate Referential Integrity

Check that every Silver order-item record has a matching order and product dimension record.

In [0]:
missing_orders = (
    silver_items
    .join(
        silver_orders.select("order_id"),
        on="order_id",
        how="left_anti"
    )
    .count()
)

add_check("Order items without matching orders", missing_orders)

missing_products = (
    silver_items
    .join(
        silver_products.select("product_id"),
        on="product_id",
        how="left_anti"
    )
    .count()
)

add_check("Order items without matching products", missing_products)

## 7. Validate Gold-Layer Metrics

Confirm that calculated rates remain between 0 and 1 and that basket-size metrics are positive.

In [0]:
invalid_customer_reorder_rates = gold_customer.filter(
    (F.col("customer_reorder_rate") < 0) |
    (F.col("customer_reorder_rate") > 1)
).count()

add_check(
    "Invalid customer reorder rates",
    invalid_customer_reorder_rates
)

invalid_product_reorder_rates = gold_product.filter(
    (F.col("reorder_rate") < 0) |
    (F.col("reorder_rate") > 1)
).count()

add_check(
    "Invalid product reorder rates",
    invalid_product_reorder_rates
)

invalid_basket_sizes = gold_customer.filter(
    F.col("average_basket_size") <= 0
).count()

add_check("Invalid average basket sizes", invalid_basket_sizes)

## 8. Create the Data-Quality Report

Convert the accumulated quality-check results into a Spark DataFrame and display the complete PASS/FAIL report.

In [0]:
quality_report = spark.createDataFrame(quality_results)

display(
    quality_report
    .orderBy(
        F.desc("status"),
        F.asc("check_name")
    )
)

check_name,failed_rows,severity,status
Duplicate order IDs,0,ERROR,PASS
Duplicate product IDs,0,ERROR,PASS
Invalid average basket sizes,0,ERROR,PASS
Invalid cart positions,0,ERROR,PASS
Invalid customer reorder rates,0,ERROR,PASS
Invalid day-of-week values,0,ERROR,PASS
Invalid order-hour values,0,ERROR,PASS
Invalid product reorder rates,0,ERROR,PASS
Invalid reordered values,0,ERROR,PASS
Missing product names,0,ERROR,PASS


## 9. Persist the Quality Results

Store the quality-check output as a managed Delta table so it can be audited, displayed, and reused in downstream reporting.

In [0]:
quality_report.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_data_quality_report")

## 10. Summarize Quality Status

Aggregate the detailed checks by status to show the number of checks executed and the total number of failed rows.

In [0]:
quality_summary = (
    quality_report
    .groupBy("status")
    .agg(
        F.count("*").alias("number_of_checks"),
        F.sum("failed_rows").alias("total_failed_rows")
    )
)

display(quality_summary)

status,number_of_checks,total_failed_rows
PASS,16,0


## Data Quality Summary

The Silver and Gold layers were validated for:

- duplicate identifiers
- missing required values
- invalid business ranges
- broken order and product relationships
- invalid reorder-rate and basket-size metrics

The results were stored in `gold_data_quality_report`.

This quality layer improves trust in the downstream business analysis and dashboard outputs.